# Semantic Chunking

- SemanticChunker is a document splitter that uses embedding similarity between sentences to decide chunk boundries.
- It ensures that each chunk is semantically coherent and not cut off mid-thought like traditional character/token splitter.

In [2]:
import os
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

load_dotenv(override=True)

True

In [3]:
model = SentenceTransformer(
    os.getenv("HUGGING_FACE_EMBEDDING_MODEL")
)

model

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10722.59it/s]


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)

In [4]:
## Sample text

text = """
    The cat sat quietly on the windowsill, watching birds in the garden.
    Artificial intelligence is transforming how businesses make decisions.
    She poured herself a cup of coffee and opened her laptop.
    The stock market experienced significant volatility last quarter.
    Climate change poses one of the greatest challenges to modern civilization.
    He learned to play the piano at the age of seven.
    The restaurant serves authentic Italian cuisine in a cozy atmosphere.
    Machine learning models require large amounts of training data.
    The hikers reached the mountain summit just before sunset.
    Online shopping has fundamentally changed consumer behavior.
    The novel explores themes of love, loss, and redemption.
    Regular exercise improves both physical and mental health.
    The spacecraft successfully landed on the surface of Mars.
    She speaks four languages fluently, including Mandarin and Spanish.
    The company announced a new sustainability initiative to reduce carbon emissions.
    Children learn best through play and hands-on exploration.
    The ancient library contained thousands of rare manuscripts.
    Cybersecurity threats are becoming increasingly sophisticated.
    He proposed to his girlfriend on a beach in Hawaii.
    Renewable energy sources like solar and wind are growing rapidly.
"""

In [6]:
## Step 1: Split into sentences

sentences = [s.strip() for s in text.split("\n") if s.strip()]

sentences

['The cat sat quietly on the windowsill, watching birds in the garden.',
 'Artificial intelligence is transforming how businesses make decisions.',
 'She poured herself a cup of coffee and opened her laptop.',
 'The stock market experienced significant volatility last quarter.',
 'Climate change poses one of the greatest challenges to modern civilization.',
 'He learned to play the piano at the age of seven.',
 'The restaurant serves authentic Italian cuisine in a cozy atmosphere.',
 'Machine learning models require large amounts of training data.',
 'The hikers reached the mountain summit just before sunset.',
 'Online shopping has fundamentally changed consumer behavior.',
 'The novel explores themes of love, loss, and redemption.',
 'Regular exercise improves both physical and mental health.',
 'The spacecraft successfully landed on the surface of Mars.',
 'She speaks four languages fluently, including Mandarin and Spanish.',
 'The company announced a new sustainability initiative t

In [7]:
### Step 2: Embed each sentence
embeddings = model.encode(sentences)

### Step 3: Initialize parameters
threshold = 0.7 # Control chunks tightness
chunks = []
current_chunks = [sentences[0]]

### Step 4: Semantic grouping based on threshold
for i in range(1, len(sentences)):
    sim = cosine_similarity(
        [embeddings[i-1]],
        [embeddings[i]]
    )[0][0]

    if sim >= threshold:
        current_chunks.append(sentences[i])
    else:
        chunks.append(" ".join(current_chunks))
        current_chunks = [sentences[i]]

# Append the last chunks
chunks.append(" ".join(current_chunks))

# Output the chunks
print("\n📌 Semantic chunks:")
for idx, chunk in enumerate(chunks):
    print(f"\n Chunks {idx+1}:\n{chunks}")


📌 Semantic chunks:

 Chunks 1:
['The cat sat quietly on the windowsill, watching birds in the garden.', 'Artificial intelligence is transforming how businesses make decisions.', 'She poured herself a cup of coffee and opened her laptop.', 'The stock market experienced significant volatility last quarter.', 'Climate change poses one of the greatest challenges to modern civilization.', 'He learned to play the piano at the age of seven.', 'The restaurant serves authentic Italian cuisine in a cozy atmosphere.', 'Machine learning models require large amounts of training data.', 'The hikers reached the mountain summit just before sunset.', 'Online shopping has fundamentally changed consumer behavior.', 'The novel explores themes of love, loss, and redemption.', 'Regular exercise improves both physical and mental health.', 'The spacecraft successfully landed on the surface of Mars.', 'She speaks four languages fluently, including Mandarin and Spanish.', 'The company announced a new sustainab

## RAG Pipeline Modular coding

In [8]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.documents import Document
from langchain_classic.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chat_models import init_chat_model
from langchain_classic.schema.runnable import RunnableLambda, RunnableMap
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
import os

load_dotenv(override=True)

True

In [20]:
### Custom sementic chunker with threshold
class ThresholdSemanticChunker:

    def __init__(self, model_name, threshold=0.7):
        self.model = SentenceTransformer(model_name)
        self.threshold = threshold

    def split(self, text: str):
        sentences = [s.strip() for s in text.split(".") if s.strip()]
        embeddings = self.model.encode(sentences)
        chunks = []
        current_chunks = [sentences[0]]

        for i in range(1, len(sentences)):
            sim = cosine_similarity([embeddings[i-1]], [embeddings[i]])[0][0]
            if sim >= self.threshold:
                current_chunks.append(sentences[i])
            else:
                chunks.append(". ".join(current_chunks)+".")
                current_chunks = [sentences[i]]

        chunks.append(". ".join(current_chunks) + ".")
        return chunks

    def split_documents(self, docs):
        results = []
        for doc in docs:
            for chunk in self.split(doc.page_content):
                results.append(Document(page_content=chunk, metadata=doc.metadata))

        return results

In [21]:
doc = Document(page_content=text)

doc

Document(metadata={}, page_content='\n    The cat sat quietly on the windowsill, watching birds in the garden.\n    Artificial intelligence is transforming how businesses make decisions.\n    She poured herself a cup of coffee and opened her laptop.\n    The stock market experienced significant volatility last quarter.\n    Climate change poses one of the greatest challenges to modern civilization.\n    He learned to play the piano at the age of seven.\n    The restaurant serves authentic Italian cuisine in a cozy atmosphere.\n    Machine learning models require large amounts of training data.\n    The hikers reached the mountain summit just before sunset.\n    Online shopping has fundamentally changed consumer behavior.\n    The novel explores themes of love, loss, and redemption.\n    Regular exercise improves both physical and mental health.\n    The spacecraft successfully landed on the surface of Mars.\n    She speaks four languages fluently, including Mandarin and Spanish.\n    T

In [26]:
## Chunking

chunker = ThresholdSemanticChunker(os.getenv("HUGGING_FACE_EMBEDDING_MODEL"), threshold=0.7)
chunks = chunker.split_documents([doc])
chunks

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 20263.29it/s]


[Document(metadata={}, page_content='The cat sat quietly on the windowsill, watching birds in the garden.'),
 Document(metadata={}, page_content='Artificial intelligence is transforming how businesses make decisions.'),
 Document(metadata={}, page_content='She poured herself a cup of coffee and opened her laptop.'),
 Document(metadata={}, page_content='The stock market experienced significant volatility last quarter.'),
 Document(metadata={}, page_content='Climate change poses one of the greatest challenges to modern civilization.'),
 Document(metadata={}, page_content='He learned to play the piano at the age of seven.'),
 Document(metadata={}, page_content='The restaurant serves authentic Italian cuisine in a cozy atmosphere.'),
 Document(metadata={}, page_content='Machine learning models require large amounts of training data.'),
 Document(metadata={}, page_content='The hikers reached the mountain summit just before sunset.'),
 Document(metadata={}, page_content='Online shopping has 

In [27]:
len(chunks)

20

In [28]:
### Vector store
embedding = HuggingFaceEmbeddings(
    model_name = os.getenv("HUGGING_FACE_EMBEDDING_MODEL")
)
vector_store = FAISS.from_documents(chunks, embedding)
retriever = vector_store.as_retriever()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13312.38it/s]


In [29]:
### Prompt Template

template = """Answer the question based on the following context:
{context}

Question: {question}
"""

prompt = PromptTemplate.from_template(template)

prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based on the following context:\n{context}\n\nQuestion: {question}\n')

In [30]:
### LLM
llm = init_chat_model(model=os.getenv("OLLAMA_CHAT_MODEL"), temperature=0.5)

rag_chain = (
    RunnableMap(
        {
            "context": lambda x: retriever.invoke(x['question']),
            "question": lambda x: x["question"],
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)

In [32]:
query = {"question": "What is AI?"}
result = rag_chain.invoke(query)

print(result)

Based on the provided context, the documents state that "Artificial intelligence is transforming how businesses make decisions," but they do not provide a definition of what AI is.
